In [4]:
import os
import cv2
import glob
import random
import numpy as np
from tqdm import tqdm
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, Conv2DTranspose, 
                                     concatenate, BatchNormalization, Activation, 
                                     add, GlobalAveragePooling2D, Reshape, Dense, 
                                     Multiply, MultiHeadAttention, LayerNormalization)
from sklearn.model_selection import train_test_split
from IPython.display import Markdown, display

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')

# Thiết lập Seed cho khâu lấy mẫu dữ liệu cố định
SEED_VALUE = 24
random.seed(SEED_VALUE)
np.random.seed(SEED_VALUE)
tf.random.set_seed(SEED_VALUE)

# BỎ CHẾ ĐỘ TUẦN TỰ ĐỂ GIẢI PHÓNG TỐC ĐỘ GPU CHẠY SONG SONG TỐI ĐA
if 'TF_DETERMINISTIC_OPS' in os.environ:
    del os.environ['TF_DETERMINISTIC_OPS']

print("TensorFlow version:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
print("⚡ Đã kích hoạt luồng xử lý Async song song giúp tăng tốc độ train!")

TensorFlow version: 2.19.0
Num GPUs Available: 2
⚡ Đã kích hoạt luồng xử lý Async song song giúp tăng tốc độ train!


In [5]:
IMG_HEIGHT, IMG_WIDTH, N_CHANNELS = 192, 256, 3       
BATCH_SIZE, EPOCHS, ALPHA, LEARNING_RATE = 8, 150, 1.67, 1e-3   

image_dir = "/kaggle/input/datasets/quanhh42/multiresunet-datasets/ISIC2018_Task1-2_Training_Input/ISIC2018_Task1-2_Training_Input"
mask_dir = "/kaggle/input/datasets/quanhh42/multiresunet-datasets/ISIC2018_Task1_Training_GroundTruth/ISIC2018_Task1_Training_GroundTruth"

def load_isic2018_sampled_data(num_samples=500):
    print(f"\nĐang quét thư mục và nạp ngẫu nhiên {num_samples} cặp ảnh ISIC 2018...")
    image_paths = sorted(glob.glob(os.path.join(image_dir, '*.jpg')))
    mask_paths = sorted(glob.glob(os.path.join(mask_dir, '*.png')))
    
    image_dict = {os.path.basename(p).replace('.jpg', ''): p for p in image_paths}
    mask_dict = {os.path.basename(p).replace('_segmentation.png', ''): p for p in mask_paths}
    common_stems = sorted(list(set(image_dict.keys()).intersection(set(mask_dict.keys()))))
    
    if len(common_stems) == 0:
        raise ValueError("Lỗi! Không tìm thấy dữ liệu ảnh tương thích.")
        
    random.shuffle(common_stems)
    sampled_stems = common_stems[:num_samples]
    
    X, Y = [], []
    for stem in tqdm(sampled_stems, desc="Loading & Normalizing Images"):
        img = cv2.imread(image_dict[stem], cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT)) / 255.0
        X.append(img)
        
        msk = cv2.imread(mask_dict[stem], cv2.IMREAD_GRAYSCALE)
        msk = cv2.resize(msk, (IMG_WIDTH, IMG_HEIGHT)) / 255.0
        Y.append(np.round(msk, 0))
        
    return np.array(X, dtype=np.float32), np.expand_dims(np.array(Y, dtype=np.float32), -1)

try:
    X, Y = load_isic2018_sampled_data(num_samples=500)
    print(f"Bộ dữ liệu trích mẫu sẵn sàng! X shape: {X.shape}, Y shape: {Y.shape}")
except Exception as e:
    print(f"Thông báo tải file: {e} -> Kích hoạt mảng giả lập dự phòng.")
    X = np.random.rand(500, IMG_HEIGHT, IMG_WIDTH, 3).astype(np.float32)
    Y = np.random.randint(0, 2, (500, IMG_HEIGHT, IMG_WIDTH, 1)).astype(np.float32)

# Phân chia dữ liệu Train-Test bằng train_test_split (80/20) để tăng tốc độ bóc tách thành phần
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=SEED_VALUE)
print(f"✔️ Train-Test Split cố định: Train = {X_train.shape[0]} ảnh | Val = {X_val.shape[0]} ảnh")


Đang quét thư mục và nạp ngẫu nhiên 500 cặp ảnh ISIC 2018...


Loading & Normalizing Images: 100%|██████████| 500/500 [01:06<00:00,  7.57it/s]


Bộ dữ liệu trích mẫu sẵn sàng! X shape: (500, 192, 256, 3), Y shape: (500, 192, 256, 1)
✔️ Train-Test Split cố định: Train = 400 ảnh | Val = 100 ảnh


In [6]:
# --- HÀM TỔN THẤT VÀ CHỈ SỐ ĐÁNH GIÁ ---
def jacard(y_true, y_pred):
    y_true_f, y_pred_f = K.flatten(y_true), K.flatten(y_pred)
    return K.sum(y_true_f * y_pred_f) / (K.sum(y_true_f + y_pred_f - y_true_f * y_pred_f) + K.epsilon())

def dice_coef(y_true, y_pred):
    y_true_f, y_pred_f = K.flatten(y_true), K.flatten(y_pred)
    return (2.0 * K.sum(y_true_f * y_pred_f) + K.epsilon()) / (K.sum(y_true_f) + K.sum(y_pred_f) + K.epsilon())

def ea_ftl_loss(y_true, y_pred):
    y_true_f, y_pred_f = K.flatten(y_true), K.flatten(y_pred)
    tp = K.sum(y_true_f * y_pred_f)
    fp = K.sum((1.0 - y_true_f) * y_pred_f)
    fn = K.sum(y_true_f * (1.0 - y_pred_f))
    tversky = (tp + K.epsilon()) / (tp + 0.3 * fp + 0.7 * fn + K.epsilon())
    ftl = K.pow((1.0 - tversky), 4./3.)
    y_true_edges = tf.image.sobel_edges(y_true)
    y_pred_edges = tf.image.sobel_edges(y_pred)
    edge_loss = K.mean(K.abs(y_true_edges - y_pred_edges))
    return ftl + 0.5 * edge_loss

def conv2d_bn(x, filters, num_row, num_col, padding='same', strides=(1, 1), activation='relu'):
    x = Conv2D(filters, (num_row, num_col), strides=strides, padding=padding, use_bias=False)(x)
    x = BatchNormalization(axis=3, scale=False)(x)
    return Activation(activation)(x) if activation is not None else x

# --- CÁC KHỐI PHẦN TỬ LIÊN LINH HOẠT PHỤC VỤ CHẠY LUỒNG ĐỘNG ---
def MultiResBlock_factory(U, inp, use_se=False): 
    W = ALPHA * U
    short = conv2d_bn(inp, int(W*0.167)+int(W*0.333)+int(W*0.5), 1, 1, activation=None)
    c3 = conv2d_bn(inp, int(W*0.167), 3, 3); c5 = conv2d_bn(c3, int(W*0.333), 3, 3); c7 = conv2d_bn(c5, int(W*0.5), 3, 3)
    out = Activation('relu')(add([short, concatenate([c3, c5, c7], 3)]))
    if use_se:
        c_se = K.int_shape(out)[3]
        se = GlobalAveragePooling2D()(out)
        se = Reshape((1, 1, c_se))(se)
        se = Dense(c_se // 8, activation='relu', use_bias=False)(se)
        se = Dense(c_se, activation='sigmoid', use_bias=False)(se)
        out = Multiply()([out, se])
    return out

def ResPath_factory(f, length, inp, use_se=False, use_att_respath=False): 
    out = conv2d_bn(inp, f, 3, 3)
    out = add([conv2d_bn(inp, f, 1, 1, activation=None), out])
    out = Activation('relu')(BatchNormalization(axis=3)(out))
    for _ in range(length - 1):
        short = conv2d_bn(out, f, 1, 1, activation=None)
        out = Activation('relu')(add([short, conv2d_bn(out, f, 3, 3)]))
        out = BatchNormalization(axis=3)(out)
        
    # Nếu kích hoạt cờ Att-ResPath (Đường truyền skip có cơ chế Attention)
    if use_att_respath:
        c_att = K.int_shape(out)[3]
        gating = Conv2D(c_att, (1, 1), padding='same', activation='sigmoid')(out)
        out = Multiply()([out, gating])
        
    if use_se and not use_att_respath: # Tiêu chuẩn hóa SE độc lập nếu không dính cấu hình Att-ResPath
        c_se = K.int_shape(out)[3]
        se = GlobalAveragePooling2D()(out)
        se = Reshape((1, 1, c_se))(se)
        se = Dense(c_se // 8, activation='relu', use_bias=False)(se)
        se = Dense(c_se, activation='sigmoid', use_bias=False)(se)
        out = Multiply()([out, se])
    return out

def TransformerBlock_factory(inputs):
    shape = K.int_shape(inputs); h, w, c = shape[1], shape[2], shape[3]
    x = Reshape((h * w, c))(inputs)
    attn_out = MultiHeadAttention(num_heads=4, key_dim=128)(x, x)
    x = LayerNormalization(epsilon=1e-6)(add([x, attn_out]))
    ffn_out = Dense(c)(Dense(512, activation='relu')(x))
    return Reshape((h, w, c))(LayerNormalization(epsilon=1e-6)(add([x, ffn_out])))

os.makedirs('checkpoint_ablation_isic', exist_ok=True)
print("✔️ Toàn bộ tài nguyên nền tảng phục vụ Att-ResPath đã sẵn sàng.")

✔️ Toàn bộ tài nguyên nền tảng phục vụ Att-ResPath đã sẵn sàng.


In [7]:
# Cấu trúc hàm sinh mô hình linh hoạt động cho ma trận Ablation Matrix
def build_ablation_network(use_se, use_transformer, use_att_respath):
    inputs = Input((IMG_HEIGHT, IMG_WIDTH, 3))
    
    # Encoder
    m1 = MultiResBlock_factory(32, inputs, use_se); p1 = MaxPooling2D((2,2))(m1); r1 = ResPath_factory(32, 4, m1, use_se, use_att_respath)
    m2 = MultiResBlock_factory(64, p1, use_se); p2 = MaxPooling2D((2,2))(m2); r2 = ResPath_factory(64, 3, m2, use_se, use_att_respath)
    m3 = MultiResBlock_factory(128, p2); p3 = MaxPooling2D((2,2))(m3); r3 = ResPath_factory(128, 2, m3, use_se, use_att_respath)
    m4 = MultiResBlock_factory(256, p3); p4 = MaxPooling2D((2,2))(m4); r4 = ResPath_factory(256, 1, m4, use_se, use_att_respath)
    
    # Bottleneck
    m5 = MultiResBlock_factory(512, p4)
    if use_transformer:
        m5 = TransformerBlock_factory(m5)
        
    # Decoder
    u6 = concatenate([Conv2DTranspose(256, (2,2), strides=(2,2), padding='same')(m5), r4], 3); m6 = MultiResBlock_factory(256, u6)
    u7 = concatenate([Conv2DTranspose(128, (2,2), strides=(2,2), padding='same')(m6), r3], 3); m7 = MultiResBlock_factory(128, u7)
    u8 = concatenate([Conv2DTranspose(64, (2,2), strides=(2,2), padding='same')(m7), r2], 3); m8 = MultiResBlock_factory(64, u8, use_se)
    u9 = concatenate([Conv2DTranspose(32, (2,2), strides=(2,2), padding='same')(m8), r1], 3); m9 = MultiResBlock_factory(32, u9, use_se)
    
    return Model(inputs, Conv2D(1, (1,1), activation='sigmoid')(m9))

# Mở rộng danh mục lên 9 nhánh thử nghiệm cấu hình có sự góp mặt độc lập của Att-ResPath
ablation_tasks = [
    {"name": "Baseline MultiResUNet",                  "se": False, "trans": False, "att_res": False, "loss_type": "bce"},
    {"name": "Baseline + Transformer",                 "se": False, "trans": True,  "att_res": False, "loss_type": "bce"},
    {"name": "Baseline + SE",                          "se": True,  "trans": False, "att_res": False, "loss_type": "bce"},
    {"name": "Baseline + Att-ResPath",                 "se": False, "trans": False, "att_res": True,  "loss_type": "bce"},
    {"name": "Baseline + EA-FTL Loss",                 "se": False, "trans": False, "att_res": False, "loss_type": "ftl"},
    {"name": "Baseline + SE + Transformer",            "se": True,  "trans": True,  "att_res": False, "loss_type": "bce"},
    {"name": "Baseline + SE + EA-FTL Loss",            "se": True,  "trans": False, "att_res": False, "loss_type": "ftl"},
    {"name": "Baseline + Transformer + EA-FTL Loss",    "se": False, "trans": True,  "att_res": False, "loss_type": "ftl"},
    {"name": "Full Proposed Model (HTS-MultiResUNet)", "se": True,  "trans": True,  "att_res": True,  "loss_type": "ftl"}
]

ablation_results = []

for idx, task in enumerate(ablation_tasks, 1):
    print("\n" + "="*80)
    print(f" 🚀 ĐANG CHẠY CẤU HÌNH {idx}/{len(ablation_tasks)}: {task['name']} (Train-Test Split)")
    print("="*80)
    
    K.clear_session()
    model = build_ablation_network(use_se=task["se"], use_transformer=task["trans"], use_att_respath=task["att_res"])
    
    current_loss = 'binary_crossentropy' if task["loss_type"] == "bce" else ea_ftl_loss
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss=current_loss, metrics=[jacard, dice_coef])
    
    chkpt_path = f"checkpoint_ablation_isic/task_{idx}_traintest.weights.h5"
    chkpt = tf.keras.callbacks.ModelCheckpoint(chkpt_path, monitor='val_jacard', mode='max', save_best_only=True, save_weights_only=True, verbose=0)
    
    # Huấn luyện liên tục không ngắt quãng trọn vẹn 150 Epochs
    history = model.fit(X_train, Y_train, validation_data=(X_val, Y_val), batch_size=BATCH_SIZE, epochs=EPOCHS, callbacks=[chkpt], verbose=1)
    
    best_j = max(history.history['val_jacard']) * 100
    best_d = max(history.history['val_dice_coef']) * 100
    
    ablation_results.append({
        "name": task["name"],
        "se": "✓" if task["se"] else "✗",
        "trans": "✓" if task["trans"] else "✗",
        "att_res": "✓" if task["att_res"] else "✗",
        "loss": "EA-FTL" if task["loss_type"] == "ftl" else "BCE",
        "jaccard": f"{best_j:.2f}",
        "dice": f"{best_d:.2f}"
    })
    print(f"✔️ Xong nhánh {idx}! Jaccard = {best_j:.2f}% | Dice = {best_d:.2f}%")

# --- SINH BẢNG SO SÁNH ĐỊNH LƯỢNG HỌC THUẬT HOÀN CHỈNH ---
print("\n📊 TIẾN TRÌNH KẾT THÚC! BẢNG SỐ LIỆU NGHIÊN CỨU THÀNH PHẦN ABLATION STUDY (ISIC 2018):")

markdown_table = """| Configuration | SE-Block | Transformer | Att-ResPath | EA-FTL Loss | Jaccard (%) ↑ | Dice (%) ↑ |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
"""
for res in ablation_results:
    markdown_table += f"| {res['name']} | {res['se']} | {res['trans']} | {res['att_res']} | {res['loss']} | {res['jaccard']} | {res['dice']} |\n"

display(Markdown(markdown_table))


 🚀 ĐANG CHẠY CẤU HÌNH 1/9: Baseline MultiResUNet (Train-Test Split)


I0000 00:00:1782012226.649871      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782012226.655963      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/150


I0000 00:00:1782012251.932747     140 service.cc:152] XLA service 0x7f172c001e60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782012251.932803     140 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1782012251.932810     140 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1782012256.252305     140 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-06-21 03:24:24.841276: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-21 03:24:25.013729: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-21 03:24:28.864141: E external/local_xl

50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 273ms/step - dice_coef: 0.4563 - jacard: 0.3046 - loss: 0.4028

2026-06-21 03:25:31.784604: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-21 03:25:31.955943: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


50/50 ━━━━━━━━━━━━━━━━━━━━ 106s 582ms/step - dice_coef: 0.4577 - jacard: 0.3058 - loss: 0.4020 - val_dice_coef: 0.3932 - val_jacard: 0.2495 - val_loss: 4.3578
Epoch 2/150
50/50 ━━━━━━━━━━━━━━━━━━━━ 15s 299ms/step - dice_coef: 0.6119 - jacard: 0.4457 - loss: 0.2818 - val_dice_coef: 0.3371 - val_jacard: 0.2078 - val_loss: 0.7902
Epoch 3/150
50/50 ━━━━━━━━━━━━━━━━━━━━ 16s 318ms/step - dice_coef: 0.6196 - jacard: 0.4535 - loss: 0.2707 - val_dice_coef: 0.4660 - val_jacard: 0.3081 - val_loss: 1.0133
Epoch 4/150
50/50 ━━━━━━━━━━━━━━━━━━━━ 16s 324ms/step - dice_coef: 0.6346 - jacard: 0.4697 - loss: 0.2599 - val_dice_coef: 0.4834 - val_jacard: 0.3264 - val_loss: 0.7539
Epoch 5/150
50/50 ━━━━━━━━━━━━━━━━━━━━ 17s 331ms/step - dice_coef: 0.6526 - jacard: 0.4900 - loss: 0.2476 - val_dice_coef: 0.5058 - val_jacard: 0.3424 - val_loss: 1.6067
Epoch 6/150
50/50 ━━━━━━━━━━━━━━━━━━━━ 17s 334ms/step - dice_coef: 0.6664 - jacard: 0.5057 - loss: 0.2412 - val_dice_coef: 0.5744 - val_jacard: 0.4086 - val_loss

| Configuration | SE-Block | Transformer | Att-ResPath | EA-FTL Loss | Jaccard (%) ↑ | Dice (%) ↑ |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| Baseline MultiResUNet | ✗ | ✗ | ✗ | BCE | 75.41 | 85.79 |
| Baseline + Transformer | ✗ | ✓ | ✗ | BCE | 71.99 | 83.54 |
| Baseline + SE | ✓ | ✗ | ✗ | BCE | 75.97 | 86.07 |
| Baseline + Att-ResPath | ✗ | ✗ | ✓ | BCE | 76.18 | 86.30 |
| Baseline + EA-FTL Loss | ✗ | ✗ | ✗ | EA-FTL | 76.07 | 86.18 |
| Baseline + SE + Transformer | ✓ | ✓ | ✗ | BCE | 73.75 | 84.61 |
| Baseline + SE + EA-FTL Loss | ✓ | ✗ | ✗ | EA-FTL | 78.52 | 87.81 |
| Baseline + Transformer + EA-FTL Loss | ✗ | ✓ | ✗ | EA-FTL | 72.88 | 84.06 |
| Full Proposed Model (HTS-MultiResUNet) | ✓ | ✓ | ✓ | EA-FTL | 75.63 | 85.67 |
